# 模块大作业：电商履约异常追踪台

## 任务来函

客服团队发现延期投诉增加，但现有订单、客户、商品和明细表彼此分散。你的任务是建立一张订单粒度事实表，识别延期风险集中的地区或品类，并输出可以交给客服复核的异常订单工单。

> 这是一份需要由你继续完成的项目 Notebook。系统只提供任务、里程碑和少量代码起点；请自行新增 Markdown 与代码单元，保留关键输出，并解释你的选择。

## 你已经拥有的项目零件

| 已学阶段 | 章节 | 可带入作业的项目零件 |
| --- | --- | --- |
| 读懂与清洗 | 第 22–28 章 | 字段检查、类型转换、缺失/重复处理与文件读写 |
| 建立指标 | 第 29 章 | 分组、聚合和透视结果 |
| 形成事实表 | 第 30 章 | 多表连接、粒度验证与结构转换 |
| 追踪变化 | 第 31 章 | 排序、累计、滚动和排名指标 |

模块作业不是重新开始。请从前面章节选择可复用的规则、数据结构、分析表、图表草稿或验证方法，并在新增的 Markdown 单元中写明“复用了什么、做了什么调整”。


## 学习目标

完成“模块大作业：电商履约异常追踪台”后，你应该能够：

1. 能把项目拆成数据质量、核心分析、结果表达和交付复现四个阶段。
2. 能为每个阶段留下可核对的中间结果和一句解释。
3. 能交付可从头运行的 Notebook，并写清限制与下一步。


## 运行后应观察

从头运行后，应在四个阶段分别留下质量检查、核心处理结果、证据表或图，以及可复现交付说明。

如果暂时没有输出，先确认是否按顺序运行了前置单元格；再检查变量名、数据形状或路径，而不是直接跳到参考实现。


## 本章目标

| 完成后能够 | 对应完成证据 |
|---|---|
| 定义订单、客户和物流表的粒度、主键及清洗规则，记录数据质量变化。 | 里程碑 1：字段合同、缺失与重复统计、保留和剔除记录的数量及理由。 |
| 在不放大订单粒度的前提下合并多表，并正确计算分组与时间窗口指标。 | 里程碑 2：合并前后行数、主键唯一性、未匹配记录以及先排序再计算的履约指标。 |
| 把分析结果整理为可追踪的异常工单，并验证导出后关键字段和口径不变。 | 里程碑 3：含订单标识和原因的异常清单、事实表和指标表，以及导出重读对照。 |


## 学习准备与补学路径

先独立尝试下面的小任务。遇到困难时回看对应章节，再返回当前里程碑；它们不另设章节作业，也不单独计分。

| 遇到的问题 | 回看章节 | 再做一次 |
|---|---|---|
| 清洗后无法解释少了哪些记录 | [第26章 数据质量检查与清洗](/course/chapter-26) | 对含缺失和重复的小表分别记录处理前后行数，保留剔除原因。 |
| 合并后订单数或金额膨胀 | [第30章 数据合并与结构转换](/course/chapter-30) | 先检查两表连接键是否唯一，使用匹配的 validate 约束并统计未匹配键。 |
| 窗口指标混入其他对象或未来信息 | [第31章 窗口计算与探索性分析](/course/chapter-31) | 按对象和日期排序，用两组短序列核对分组窗口；预测场景另检查是否需要 shift。 |


## 任务合同：数据、边界与交付

**数据与边界：** 使用课程 Olist 订单、订单明细、客户、商品及类别翻译数据。先写明每张表一行代表什么、主键和连接关系；原始文件只读，清洗后的事实表另行导出。

**必须完成：**

1. 记录每张源表的粒度、主键、类型、缺失和重复情况。
2. 至少合并三张表，并验证合并没有意外放大订单粒度。
3. 构造延期时长、订单金额、商品数等必要字段。
4. 使用分组指标和一个排序后的窗口指标追踪履约变化。
5. 输出订单级异常工单，并为发现写明样本量、口径和局限。

**最终交付：**

- 数据字典、合并关系和质量审计。
- 订单粒度事实表、履约趋势指标表和异常工单 CSV。
- 3 条可回溯到指标或订单的运营发现。


## 如何开始：三级提示

### 第一层｜操作路线

1. 读取后立即检查行数、字段、类型和缺失。
2. 先确认主键与粒度，再进行 `merge`。
3. 转换日期后再做 `groupby`、`rolling` 或 `rank`。

### 第二层｜代码起点

下面只给出 API 或结构起点，字段、参数、规则、异常处理和结果解释均由你完成。

```python
import pandas as pd

orders = pd.read_csv("...csv")
orders["...date"] = pd.to_datetime(orders["...date"], errors="coerce")
# TODO：选择主键合并，并检查合并前后的行数
```

### 第三层｜遇到问题时检查

先检查输入数据与中间结果，再检查字段、shape、粒度、排序或指标口径。不要通过删除校验条件来让结果“看起来正确”。


## 里程碑 1｜签订数据合同并审计源表

**承接章节：** 第 22–28 章：DataFrame、类型、日期、缺失与读写

**此刻的项目情境：** 客服团队发现延期投诉增多，但订单、客户、商品和支付表的粒度并不相同。你必须先证明数据能够支持履约追踪。

**你要解决的问题：** 每张表一行代表什么、主键是什么、哪些缺失或异常会影响延期判断？

**完成证据：** 数据字典；表级行数、类型、缺失和重复键审计；保留/排除规则。

**容易失分的地方：** 清洗后只展示最终行数，没有记录删掉了什么；把订单与明细当成相同粒度。

完成代码后，请自行新增一个 Markdown 单元，按“观察到什么 → 这说明什么 → 下一步怎么做”解释结果。


<!-- math-foundation:capstone-pandas -->
### 数学推导｜履约异常率与平均延迟

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜把日期差转成同一单位。** 每单延迟天数为 $d_i=t_i^{actual}-t_i^{promised}$。

**第 2 步｜把“是否延期”变成指标。** $l_i=\mathbf{1}(d_i>0)$，因此延期订单数是 $\sum_i l_i$。

**第 3 步｜区分两个不同问题。** $r_{late}=\sum_i l_i/n$ 回答“多少订单延期”，而

$$
\bar d_{late}=\frac{\sum_i d_i l_i}{\sum_i l_i}
$$

回答“已延期订单平均晚几天”；它与全体订单的平均日期差不是同一口径。

**把上面的关系收束为本章计算式：**

$$
r_{late}=\frac{\sum_i\mathbf{1}(d_i>0)}{n},\qquad \bar{d}=\frac{1}{n}\sum_{i=1}^{n}d_i
$$

**符号解释：** $d_i$ 是实际送达日减预计送达日的天数，$r_{late}$ 是延期订单占比。

**代码对应：** 构造 `delay_days` 和 `is_late` 后按月份、地区或品类分组聚合。

**使用边界：** 未送达订单与缺失预计日期需要单独定义，不能静默排除后仍称为总体延期率。


In [ ]:
# 里程碑 1｜签订数据合同并审计源表
# 这段代码应解决：每张表一行代表什么、主键是什么、哪些缺失或异常会影响延期判断？
# 完成后应留下：数据字典；表级行数、类型、缺失和重复键审计；保留/排除规则。
# TODO：从下面的起点继续；关键规则和设计选择需要写注释。

import pandas as pd

orders = pd.read_csv("...csv")
# TODO：为每张表记录粒度、主键、dtypes、缺失和重复键
# TODO：转换日期并保留清洗前后的数量证据


## 里程碑 2｜构建安全的订单事实表

**承接章节：** 第 29–30 章：分组聚合、合并与结构转换

**此刻的项目情境：** 客服需要一行对应一笔订单的追踪表。商品明细必须先聚合，否则金额、件数和订单量会被重复累计。

**你要解决的问题：** 怎样合并多表而不放大订单粒度，并构造延期天数、订单金额、商品数等字段？

**完成证据：** 合并关系说明；合并前后行数与唯一性检查；订单粒度事实表；连接损失记录。

**容易失分的地方：** 直接进行多对多合并；没有使用 validate、indicator 或主键唯一性检查。

完成代码后，请自行新增一个 Markdown 单元，按“观察到什么 → 这说明什么 → 下一步怎么做”解释结果。


In [ ]:
# 里程碑 2｜构建安全的订单事实表
# 这段代码应解决：怎样合并多表而不放大订单粒度，并构造延期天数、订单金额、商品数等字段？
# 完成后应留下：合并关系说明；合并前后行数与唯一性检查；订单粒度事实表；连接损失记录。
# TODO：从下面的起点继续；关键规则和设计选择需要写注释。

# TODO：先把订单明细聚合到 order_id 粒度
order_fact = orders.merge(
    ..., on="order_id", how="...", validate="...", indicator=True
)
# TODO：检查合并后 order_id 是否唯一，并解释未匹配记录


## 里程碑 3｜生成异常工单与趋势证据

**承接章节：** 第 29、31 章：分组指标、排序、窗口计算与导出

**此刻的项目情境：** 运营负责人不需要一堆平均数，而需要知道延期是否持续恶化、集中在哪些地区/品类，以及下一批应该复核哪些订单。

**你要解决的问题：** 怎样把分组指标、窗口趋势和订单级规则组合成可执行的异常工单？

**完成证据：** 至少两张运营指标表；一个排序后的窗口指标；异常订单清单；3 条有证据和局限的发现。

**容易失分的地方：** 滚动计算前没有按时间和分组排序；用总体平均掩盖样本量差异；建议无法回到具体订单。

完成代码后，请自行新增一个 Markdown 单元，按“观察到什么 → 这说明什么 → 下一步怎么做”解释结果。


In [ ]:
# 里程碑 3｜生成异常工单与趋势证据
# 这段代码应解决：怎样把分组指标、窗口趋势和订单级规则组合成可执行的异常工单？
# 完成后应留下：至少两张运营指标表；一个排序后的窗口指标；异常订单清单；3 条有证据和局限的发现。
# TODO：从下面的起点继续；关键规则和设计选择需要写注释。

# TODO：按时间与业务切片计算延期率/时长
# TODO：排序后计算 rolling / cumulative / rank 指标
issue_tickets = order_fact.loc[...]
# TODO：导出事实表、趋势指标和异常工单


## 交付、挑战与自查

**基础提交清单：**

- [ ] 所有日期在计算时效前已转换并检查。
- [ ] 事实表中的订单主键保持唯一。
- [ ] 合并未匹配记录和行数变化有解释。
- [ ] 窗口计算前已按正确分组和时间排序。

### 进阶挑战（可选）

更换延期阈值或时间窗口，检查高风险地区/品类是否稳定，并解释工单数量与漏报风险的取舍。

挑战任务必须建立在基础任务已经完整、可复现的前提上；不能用额外图表或复杂模型掩盖基础证据缺失。


## 评分标准（100 分）

### 共同能力：30 分

- **可复现性（10 分）**：重启内核后能按顺序运行，路径和依赖清楚。
- **证据与注释（10 分）**：关键代码说明设计原因，结论能回到具体输出。
- **边界与诚实表达（10 分）**：说明数据来源、假设、限制，不把相关性写成确定因果。

### 本模块核心能力：70 分

- **粒度与数据质量（20 分）**：主键、类型、缺失、重复和清洗影响记录完整。
- **合并安全性（20 分）**：连接关系正确，合并前后规模、唯一性和未匹配记录有验证。
- **分组与窗口指标（20 分）**：排序、分组和窗口口径正确，指标能够回答履约问题。
- **工单与导出（10 分）**：异常清单可执行，事实表和指标表可复用。

### 提交门槛

Notebook 必须能够运行；错误或异常记录不能被静默隐藏；关键结论必须可以回溯。最后的确认单元只检查摘要是否填写，不替代教师评分。

<!-- module-teaching-assessment -->
### 达标与返工规则

- 总分至少 60/100，共同能力至少 18/30，模块核心能力至少 42/70，且下列关键门槛全部满足，才视为达标。
- **本模块关键门槛：** 订单粒度、连接基数和未匹配记录必须可核对；不能因重复连接虚增分母或总量；日期异常不得冒充正常履约。
- 每项按证据给分：独立完成且处理边界为该项满分；主流程正确但证据不全为约 75%；最小流程可复现为约 60%；未完成或关键方法错误为 0–50%。教师须记录扣分依据。
- 学生作品须在不运行参考答案的情况下，重启内核并从头复现；参考答案中产生的变量、文件和输出不能作为自己的完成证据。
- 运行进度、摘要填写和勾选状态仅是学习记录，不是自动评分。关键门槛未满足时先返工再评，不用其他高分抵消。
- **可选迁移：** 改变延期阈值并重新生成工单，比较名单变化，说明取消或未签收订单如何进入分母。 迁移不另设加分，不挤占必做任务；可作为相应维度的边界或解释证据。


In [ ]:
# 提交前确认：请在完成三个里程碑后填写。
# 教师将结合代码、输出、解释和导出文件评分，不会只看本单元。
submission_summary = {
    "任务与使用者": "",
    "三个里程碑的完成证据": "",
    "最重要的结果或作品功能": "",
    "限制与下一步": "",
}

missing = [
    key
    for key, value in submission_summary.items()
    if not str(value).strip() or str(value).strip() == "..."
]
if missing:
    raise ValueError("请先填写提交摘要：" + "、".join(missing))
print("提交摘要已填写；请重启内核并从头运行，再对照评分标准检查证据。")


In [ ]:
# 参考答案｜里程碑 1：签订数据合同并审计源表
# 默认隐藏。建议先完成自己的实现，再展开对照设计选择。
# 三个答案 Cell 前后衔接；阅读时请按里程碑顺序理解变量与中间结果。

# 参考答案：电商履约异常追踪台
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("output/pandas_fulfillment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv("/datasets/olist_orders_dataset.csv")
items = pd.read_csv("/datasets/olist_order_items_dataset.csv")
customers = pd.read_csv("/datasets/olist_customers_dataset.csv")
products = pd.read_csv("/datasets/olist_products_dataset.csv")
translation = pd.read_csv("/datasets/product_category_name_translation.csv")

audit = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(frame),
            "duplicate_rows": int(frame.duplicated().sum()),
            "missing_cells": int(frame.isna().sum().sum()),
        }
        for name, frame in {
            "orders": orders,
            "items": items,
            "customers": customers,
            "products": products,
        }.items()
    ]
)
print(audit)

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for column in date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")


In [ ]:
# 参考答案｜里程碑 2：构建安全的订单事实表
# 默认隐藏。建议先完成自己的实现，再展开对照设计选择。
# 三个答案 Cell 前后衔接；阅读时请按里程碑顺序理解变量与中间结果。

product_lookup = products[["product_id", "product_category_name"]].merge(
    translation, on="product_category_name", how="left", validate="many_to_one"
)
item_detail = items.merge(
    product_lookup, on="product_id", how="left", validate="many_to_one"
)
item_detail["line_amount"] = (
    item_detail["price"] + item_detail["freight_value"]
)
order_amounts = (
    item_detail.groupby("order_id")
    .agg(
        {
            "line_amount": "sum",
            "order_item_id": "count",
            "product_category_name_english": lambda values: (
                values.mode().iat[0] if not values.mode().empty else "unknown"
            ),
        }
    )
    .reset_index()
    .rename(
        columns={
            "line_amount": "order_amount",
            "order_item_id": "item_count",
            "product_category_name_english": "main_category",
        }
    )
)

fact = orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator="customer_merge",
)
fact = fact.merge(
    order_amounts,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator="item_merge",
)
_check_1 = bool(fact["order_id"].is_unique)
print('自检 1：fact["order_id"].is_unique ->', "通过" if _check_1 else "需要检查")
if not _check_1:
    print("建议：", '请回看输入、处理步骤和预期结果。')

fact["delivery_days"] = (
    fact["order_delivered_customer_date"] - fact["order_purchase_timestamp"]
).dt.total_seconds() / 86400
fact["delay_days"] = (
    fact["order_delivered_customer_date"]
    - fact["order_estimated_delivery_date"]
).dt.total_seconds() / 86400
fact["is_late"] = fact["delay_days"].gt(0)
fact["purchase_month"] = (
    fact["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
)


In [ ]:
# 参考答案｜里程碑 3：生成异常工单与趋势证据
# 默认隐藏。建议先完成自己的实现，再展开对照设计选择。
# 三个答案 Cell 前后衔接；阅读时请按里程碑顺序理解变量与中间结果。

monthly = (
    fact.groupby("purchase_month")
    .agg(
        {
            "order_id": "nunique",
            "is_late": "mean",
            "delay_days": "mean",
            "order_amount": "sum",
        }
    )
    .reset_index()
    .rename(
        columns={
            "order_id": "orders",
            "is_late": "late_rate",
            "delay_days": "average_delay_days",
            "order_amount": "revenue",
        }
    )
    .sort_values("purchase_month")
)
monthly["late_rate_3m_avg"] = (
    monthly["late_rate"].rolling(3, min_periods=1).mean()
)

state_summary = (
    fact.groupby("customer_state")
    .agg(
        {
            "order_id": "nunique",
            "is_late": "mean",
            "delay_days": "mean",
        }
    )
    .reset_index()
    .rename(
        columns={
            "order_id": "orders",
            "is_late": "late_rate",
            "delay_days": "average_delay_days",
        }
    )
    .query("orders >= 30")
    .sort_values(["late_rate", "orders"], ascending=[False, False])
)

issue_tickets = fact.loc[
    fact["is_late"] & fact["order_delivered_customer_date"].notna(),
    [
        "order_id",
        "customer_state",
        "main_category",
        "order_purchase_timestamp",
        "delay_days",
        "order_amount",
    ],
].sort_values(["delay_days", "order_amount"], ascending=[False, False])

fact.to_csv(OUTPUT_DIR / "order_fact.csv", index=False)
monthly.to_csv(OUTPUT_DIR / "fulfillment_monthly.csv", index=False)
state_summary.to_csv(OUTPUT_DIR / "fulfillment_by_state.csv", index=False)
issue_tickets.to_csv(OUTPUT_DIR / "late_order_tickets.csv", index=False)

print("事实表规模:", fact.shape, "订单唯一:", fact["order_id"].is_unique)
print(
    "客户/商品未匹配:",
    (fact["customer_merge"] != "both").sum(),
    (fact["item_merge"] != "both").sum(),
)
print("高延期州:")
print(state_summary.head(5).round(3))
print("异常工单数:", len(issue_tickets))


## 项目工作区

以下四个单元格是可编辑的最小项目骨架。不要把整份项目塞进一个单元格；每完成一阶段，都留下一个结果和一句解释。


### 阶段 A：输入与质量

读取数据，输出形状、字段、缺失与异常记录。


In [ ]:
# TODO: 读取数据，输出形状、字段、缺失与异常记录。
# 写完后打印一个可核对的结果，并补充一句解释。


### 阶段 B：核心处理

完成清洗、特征、统计或基线，并保留中间结果。


In [ ]:
# TODO: 完成清洗、特征、统计或基线，并保留中间结果。
# 写完后打印一个可核对的结果，并补充一句解释。


### 阶段 C：结果与证据

生成一张表或图，并写一句仅由数据支持的结论。


In [ ]:
# TODO: 生成一张表或图，并写一句仅由数据支持的结论。
# 写完后打印一个可核对的结果，并补充一句解释。


### 阶段 D：交付与限制

整理交付物，说明可复现步骤、限制和下一步。


In [ ]:
# TODO: 整理交付物，说明可复现步骤、限制和下一步。
# 写完后打印一个可核对的结果，并补充一句解释。


## 诊断式自检

不要只看代码是否报错。运行后逐项填写或口头说明下列检查项；任何一项不清楚，都应回到对应的输入、处理中间结果或图表。


In [ ]:
review = {
    "阶段中间结果": "待确认",
    "交付物可复现性": "待确认",
    "限制与下一步": "待确认"
}
for item, status in review.items():
    print(f"{'待补充' if status == '待确认' else '已确认'}：{item} -> {status}")
print("完成后，用一句话写出结果支持的结论和仍存在的限制。")
